### Transformer-from-Scratch

A clean Pytorch re-implementation of the Transformer architecture described in Attention is All You Need

#### Transformer 模型代码拆解
1. Positional Encoding
2. Multi-Head Attention
3. Feed Forward Network
4. Transformer Encoder Layer
5. Transformer Encoder
6. Transformer Decoder Layer
7. Transformer Decoder
8. Transformer Model
9. mask function
10. Example usage

整体框架：

<img src="transformer.jpg" height="320">

- Input / Output Embedding：把 token id 映射到连续向量。
- Positional Encoding：补位置信息，不补语义内容。
- Encoder：负责编码输入序列，产出 `memory`。
- Decoder：负责根据目标前缀 + `memory` 生成当前表示。
- Generator：把最后的 hidden state 投影到词表 logits。

Self-Attention：同一序列内部彼此看，建模 token 和 token 的关系。

Masked Self-Attention：只能看左边，防止训练时偷看未来。

Cross-Attention：decoder 用当前状态去查 encoder 的 `memory`。

FFN：每个位置各自过一个两层 MLP，做特征加工。

MLP（多层感知机）：若干层线性映射加非线性激活组成的前馈网络

Add + Norm：保留原输入通路，同时稳定训练。

Causal Mask：本质是上三角屏蔽，防信息泄漏。

## Transformer 面经题整理

### 1 分钟讲清 Transformer

**面试短答：**

Transformer 是一种 **基于注意力、去掉循环和卷积** 的序列建模架构。标准版是 **encoder-decoder** 结构：encoder 每层包含 **Multi-Head Self-Attention + FFN**，decoder 每层包含 **Masked Self-Attention + Cross-Attention + FFN**，并配合 **残差连接和归一化**。它相对 RNN 的核心优势是：**并行性更强、长距离依赖路径更短**；核心代价是标准 self-attention 对序列长度通常有 **二次复杂度**。今天很多大模型仍然是 Transformer 变体，只是把位置编码、归一化和 attention 实现做得更强更快，比如 **RoPE、Pre-LN、RMSNorm、FlashAttention**。

**展开回答：**

如果从代码角度看，你当前工程里已经把标准 Transformer 主干拆成了几块：

- `MHA.py`：注意力主体，核心公式是 `softmax(QK^T / sqrt(d_k))V`
- `FFN.py`：逐位置的两层 MLP，负责特征加工
- `Encoder.py`：堆叠 self-attention + FFN
- `Decoder.py`：堆叠 masked self-attention + cross-attention + FFN
- `Transformer.py`：embedding -> encoder -> decoder -> generator 的完整拼装

因此面试时如果被问“Transformer 靠什么有表达能力”，标准回答不是只说 attention，而是说 **attention + FFN + 残差 + 归一化 + 位置建模** 一起构成了完整系统。

### 1）为什么 Transformer 比 RNN 更适合大规模训练？

**面试短答：**

因为 RNN 的时间步存在严格串行依赖，而 Transformer 的 self-attention 可以在整个序列上并行计算，所以更容易吃满 GPU/TPU 这类矩阵计算硬件。同时，Transformer 的长距离依赖路径更短，两位置之间一次 attention 就能直接交互，而 RNN 需要跨很多时间步传递。

**展开回答：**

RNN 在第 `t` 步的 hidden state 依赖第 `t-1` 步，因此训练时即使使用 BPTT(Backpropagation Through Time)，也很难彻底并行。Transformer 则不同，给定输入张量

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

Q、K、V 可以对所有位置一起做线性映射，然后一次性完成矩阵乘法：

$$
QK^T \in \mathbb{R}^{B \times h \times L \times L}
$$

所以训练时更像大规模矩阵运算，而不是时间步 for-loop。与此同时，RNN 中位置 `i` 和位置 `j` 的依赖路径长度大约是 `|i-j|`，而 self-attention 中两者可以直接交互，依赖路径可视为 `O(1)`。这也是为什么它更适合大规模预训练。

**但要补一句：** 并行性更强不等于没有代价。标准 attention 的 `L x L` 交互会让长序列在显存和算力上很贵，这是 Transformer 的核心瓶颈之一。

### 2）Self-Attention 的公式是什么？每一项是什么意思？

**面试短答：**

标准 self-attention 公式是：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

其中 `Q` 表示“我现在想找什么”，`K` 表示“我这里有什么信息值得你关注”，`V` 表示“如果你关注我，你最终拿走什么内容”。

**展开回答：**

设输入为

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

通过三组线性映射得到：

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

如果有 `h` 个头，通常 reshape 成：

$$
Q,K,V \in \mathbb{R}^{B \times h \times L \times d_k},\quad d_k = d_{model}/h
$$

然后：

1. `QK^T`：计算每个 query 对每个 key 的相关性分数
2. `/ sqrt(d_k)`：控制分数尺度
3. `softmax`：把分数转成注意力权重
4. `@ V`：根据权重对 value 做加权求和，得到新的表示

和你当前 `MHA.py` 里的实现是一一对应的：

```python
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
attn_weights = torch.softmax(scores, dim=-1)
output = torch.matmul(attn_weights, V)
```

<img src="Scaled_Dot_Product_Attention.png" height="280">

### 3）为什么要除以 $\sqrt{d_k}$

**面试短答：**

因为如果不缩放，`QK^T` 的数值方差会随着维度 `d_k` 增大而变大，softmax 很容易过饱和，导致梯度变差。除以 `sqrt(d_k)` 后，attention score 的尺度更稳定。

**展开回答：**

设某一对 query、key 的每个维度近似独立，且满足：

$$
\mathbb{E}[q_i] = \mathbb{E}[k_i] = 0, \quad \operatorname{Var}(q_i)=\operatorname{Var}(k_i)=1
$$

则点积

$$
q \cdot k = \sum_{i=1}^{d_k} q_i k_i
$$

的方差近似满足：

$$
\operatorname{Var}(q \cdot k) \approx d_k
$$

于是缩放后：

$$
\operatorname{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) \approx 1
$$

这会带来两个直接好处：

- softmax 输入不会因为维度变大而越来越极端
- 梯度不会因为过早饱和而变小

这也是我们在 `MHA.py` 里那句 `/ math.sqrt(self.d_k)` 的理论来源。注意，这和 `Transformer.py` 里 embedding 乘 `sqrt(d_model)` 不是一回事：前者是 **score 稳定化**，后者更像 **输入尺度设计**。

### 4）Multi-Head Attention 为什么比单头更好？

**面试短答：**

多头的好处是让模型能在不同子空间里并行关注不同类型的关系，而不是把所有关系都压到一个 attention 模式里。最终每个 head 的结果再拼接并投影回 `d_model`，表达能力更丰富。

**展开回答：**

单头 attention 只能给出一套权重模式。如果当前序列里同时存在：

- 语法依赖
- 指代关系
- 局部短语搭配
- 长距离主题关联

单头需要在一套分数里兼顾这些结构；多头则可以让不同头分别学不同模式。数学上，如果

$$
Q_i, K_i, V_i \in \mathbb{R}^{B \times L \times d_k}
$$

则每个头独立算：

$$
head_i = \operatorname{Attention}(Q_i, K_i, V_i)
$$

最后：

$$
\operatorname{MultiHead}(Q,K,V)=\operatorname{Concat}(head_1,\dots,head_h)W_O
$$

这和 `MHA.py` 里的 `_split_heads()`、每头 attention、再 `_merge_heads()` + `W_o` 是一致的。

<img src="Multi_Head_Attention.png" height="300">

**但要严谨地补一句：** 多头并不保证每个头一定学到完全不同的功能，实际训练中会有头冗余、头相似的现象；只是给了模型这种能力和空间。

### 5）Encoder 和 Decoder 有什么区别？

**面试短答：**

Encoder 负责把输入序列编码成上下文化表示；Decoder 负责在已生成前缀和 encoder 输出的条件下生成目标序列。结构上，encoder 每层只有 self-attention + FFN；decoder 每层有 masked self-attention + cross-attention + FFN。

**展开回答：**

在我们的实现里：

- `EncoderLayer`：`self_attn -> FFN`
- `DecoderLayer`：`masked self_attn -> cross_attn -> FFN`

区别主要体现在两点：

1. **decoder 自注意力要加 causal mask**
   因为它做自回归生成，当前位置不能偷看未来 token。

2. **decoder 多了 cross-attention**
   其中 query 来自 decoder 当前状态 `x`，key/value 来自 encoder 输出 `memory`。也就是：

$$
Q = xW_Q, \quad K = memory W_K, \quad V = memory W_V
$$

因此可以把 encoder 理解成“读输入”，把 decoder 理解成“边看已生成前缀边查输入资料继续写”。如果去掉 encoder 和 cross-attention，结构就更接近 GPT 这类 decoder-only 模型。

### 6）为什么 Transformer 需要位置编码？

**面试短答：**

因为 self-attention 本身只看 token 间内容相关性，不天然知道顺序。如果没有位置信息，模型很难区分“我喜欢苹果”和“苹果喜欢我”这种词一样但顺序不同的句子。

**展开回答：**

设输入 token embedding 为：

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

如果直接对 `X` 做 self-attention，那么模型看到的是一组向量之间的相似性关系，它不知道“第几个位置”这个信息。因此原始 Transformer 在输入端加入位置编码：

$$
X' = X + PE
$$

其中 `PE` 在当前 `PostionalEncoding.py` 里是固定的正弦位置编码。它的直觉是：

- 给每个位置一个唯一但连续的向量表示
- 让模型能推断绝对位置和相对位移

**面试时最好补一句：** mask 和位置编码不是一回事。mask 决定“能不能看”，位置编码决定“看到的内容处在什么顺序/位置”。

### 7）RoPE 是什么？相比绝对位置编码有什么优点？

**面试短答：**

RoPE（Rotary Positional Embedding）是一种把位置信息通过旋转直接注入到 `Q/K` 向量中的方法。它相比绝对位置编码的一个核心优点是：attention 分数天然带有相对位置信息，而且通常对长序列外推更友好。

**展开回答：**

正弦绝对位置编码的做法是 `X + PE`；RoPE 则不是把位置向量直接加到输入上，而是把 `Q`、`K` 的每一对二维分量按位置做旋转。严谨一点地写，设第 $m$ 对二维通道在位置 $p$ 和位置 $j$ 上分别是：

$$
q_m^{(p)},\; k_m^{(j)} \in \mathbb{R}^2
$$

定义对应频率下的旋转矩阵：

$$
R_m(t)=
\begin{bmatrix}
\cos(\theta_m t) & -\sin(\theta_m t) \\
\sin(\theta_m t) & \cos(\theta_m t)
\end{bmatrix}
$$

则 RoPE 后的 query / key 为：

$$
\tilde q_m^{(p)} = R_m(p) q_m^{(p)},
\qquad
\tilde k_m^{(j)} = R_m(j) k_m^{(j)}
$$

这里最关键的是它们对 attention score 的影响。对这一对二维通道，旋转后的内积为：

$$
\big(\tilde q_m^{(p)}\big)^\top \tilde k_m^{(j)}
= \big(R_m(p) q_m^{(p)}\big)^\top \big(R_m(j) k_m^{(j)}\big)
$$

利用 $(AB)^\top = B^\top A^\top$，可得：

$$
\big(\tilde q_m^{(p)}\big)^\top \tilde k_m^{(j)}
= \big(q_m^{(p)}\big)^\top R_m(p)^\top R_m(j) k_m^{(j)}
$$

而旋转矩阵满足：

$$
R_m(a)^\top = R_m(-a),
\qquad
R_m(a)^\top R_m(b)=R_m(b-a)
$$

所以进一步化简为：

$$
\big(\tilde q_m^{(p)}\big)^\top \tilde k_m^{(j)}
= \big(q_m^{(p)}\big)^\top R_m(j-p) k_m^{(j)}
$$

这一步就是核心：旋转后的内积不再分别依赖位置 $p$ 和位置 $j$，而是通过：

$$
j-p
$$

这个相对位移进入点积。把所有二维通道对的贡献加起来后，最终 attention score 就天然带上相对位置信息。所以更准确地说：**RoPE 让 attention 中的点积项通过旋转矩阵依赖于相对位移 $j-p$，而不是只依赖单独的绝对位置编号。**

如果用复数记号，同样可以把一对二维分量写成：

$$
z = x + \mathrm{i}y
$$

这时旋转就等价于乘一个相位：

$$
z \mapsto z e^{\mathrm{i}\theta_m t}
$$

这个写法更紧凑，但面试里如果要讲严谨推导，优先用上面的旋转矩阵版本更稳。

**相比绝对位置编码的优点：**

- 相对位置信息直接融入 attention 分数本身
- 更适合表达“相隔多远、谁在左谁在右”这类关系
- 在很多长上下文场景里更稳
- 不需要把一个绝对位置向量直接加到 embedding 上

**工程上常见结论：** 原始正弦 PE 很适合教学和论文主干理解；RoPE 更像现代 LLM 的主流位置建模方案。

### 8）FFN 在 Transformer 里是干什么的？

**面试短答：**

FFN 负责对每个位置的表示做逐位置、非线性的特征加工。attention 负责 token 之间的信息交互，FFN 负责每个 token 自己在特征维上的深加工。

**展开回答：**

标准 FFN 一般写成：

$$
\operatorname{FFN}(x)=W_2\sigma(W_1x+b_1)+b_2
$$

在你的 `FFN.py` 里，就是：

```python
x = self.fc1(x)
x = self.activation(x)
x = self.dropout(x)
x = self.fc2(x)
```

如果输入 shape 是：

$$
(B, L, d_{model})
$$

那么 `nn.Linear` 默认作用在最后一维上，因此 FFN 是 **position-wise** 的：它不会混不同 token 的位置，只对每个 token 的特征向量做变换。

为什么常见设计是 `d_model -> d_ff -> d_model`？因为先升维可以给非线性变换一个更大的特征空间，再压回原维度以便残差连接和后续层继续处理。

### 9）为什么要残差连接和归一化？

**面试短答：**

残差连接是为了保留原信息并改善梯度传播，归一化是为了稳定数值尺度和优化过程。attention/FFN 提供表达能力，Add + Norm 提供可训练性。

**展开回答：**

标准 Post-LN 结构里常见的是：

$$
y = \operatorname{LN}(x + F(x))
$$

其中：

- `x + F(x)` 是残差连接
- `LN` 是 LayerNorm

**残差的作用：**

- 给原始输入保留一条直通路径
- 让子层学的是“修正量”而不是从头重写表示
- 深层时更利于梯度传播

**归一化的作用：**

- 稳定每层输出尺度
- 减少数值飘移
- 让更深的网络更容易训练

这就是为什么在 `Encoder.py`、`Decoder.py` 里看到的结构不是单纯 `MHA -> FFN`，而是 `子层输出 -> Add -> Norm`。

### 10）Pre-LN 和 Post-LN 区别是什么？

**面试短答：**

Post-LN 是原始 Transformer 论文写法：`LN(x + F(x))`；Pre-LN 是先归一化再进子层：`x + F(LN(x))`。现代大模型更常用 Pre-LN，因为深层训练通常更稳定。

**展开回答：**

**Post-LN：**

$$
y = \operatorname{LN}(x + F(x))
$$

**Pre-LN：**

$$
y = x + F(\operatorname{LN}(x))
$$

两者的核心差别不是“有没有 LayerNorm”，而是 **LayerNorm 放在子层前还是残差后**。Pre-LN 通常更稳定的原因是：

- 子层输入先被规范化，梯度路径更平滑
- 深层训练时不容易出现数值不稳
- 训练大模型时更常见

但 Post-LN 的语义更直观，也更贴近原论文，所以教学代码里常常先从 Post-LN 讲起。你当前 `Encoder.py` 和 `Decoder.py` 用的就是 Post-LN 风格。

### 11）LayerNorm 和 RMSNorm 的区别？

**面试短答：**

LayerNorm 会减均值再除标准差，RMSNorm 只按均方根做缩放，不做显式去均值。RMSNorm 更简单、算子更轻，现代大模型里很常见。

**展开回答：**

**LayerNorm：**

$$
\mu = \frac{1}{d}\sum_{i=1}^d x_i, \quad
\sigma^2 = \frac{1}{d}\sum_{i=1}^d (x_i - \mu)^2
$$

$$
y_i = \gamma_i \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta_i
$$

**RMSNorm：**

$$
\operatorname{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \varepsilon}
$$

$$
y_i = \gamma_i \frac{x_i}{\operatorname{RMS}(x)}
$$

所以 RMSNorm 的差别在于：

- 不做减均值
- 通常没有像 LayerNorm 那样的显式偏置项
- 更强调“只控制尺度，不控制中心”

工程上它的优点是实现简单、数值开销略低，许多现代 LLM（如一些 LLaMA 系列变体）会更偏向 RMSNorm。

### 12）Transformer 的复杂度瓶颈在哪里？

**面试短答：**

标准 Transformer 的主要瓶颈在 self-attention 的 `L x L` 交互，时间和显存开销都会随着序列长度呈二次增长。长文本时，这通常比 FFN 更容易成为瓶颈。

**展开回答：**

如果序列长度是 `L`，每头维度是 `d_k`，那么 attention 里最重的两步是：

1. `QK^T`：

$$
(L \times d_k)(d_k \times L) \rightarrow (L \times L)
$$

2. `attn_weights @ V`：

$$
(L \times L)(L \times d_k) \rightarrow (L \times d_k)
$$

因此标准 self-attention 的时间复杂度通常记作：

$$
O(L^2 d)
$$

而且中间 attention matrix 的 shape 是：

$$
(B, h, L, L)
$$

这就是显存压力最大的来源之一。严格来说，FFN 也很贵，尤其 `d_ff` 很大时计算量不小；但当序列很长时，attention 的二次项更容易成为瓶颈。面试时最好把这句说完整：

- **短序列、大宽度时，FFN 也可能很重**
- **长序列场景下，attention 的 `L^2` 更容易成为主瓶颈**

### 13）FlashAttention 是什么，为什么快？

**面试短答：**

FlashAttention 是一种 **IO-aware 的 exact attention 实现**。它没有改 attention 的数学定义，而是通过分块（tiling）和在线 softmax，避免显式存储完整的 $L \times L$ attention matrix，从而显著减少高带宽显存（HBM）读写，所以更快也更省显存。

**先从标准 attention 公式出发：**

$$
S = \frac{QK^T}{\sqrt{d_k}},
\qquad
P = \operatorname{softmax}(S),
\qquad
O = PV
$$

其中如果单头下 $Q, K, V \in \mathbb{R}^{L \times d_k}$，则：

$$
S, P \in \mathbb{R}^{L \times L},
\qquad
O \in \mathbb{R}^{L \times d_k}
$$

**标准实现为什么慢：**

朴素实现通常是：

1. 先算完整 score matrix $S = QK^T / \sqrt{d_k}$
2. 把 $S$ 写回 HBM (大容量但访问代价比片上 SRAM 更高的存储层)
3. 对 $S$ 做 mask 和 softmax，得到完整 $P$
4. 再把 $P$ 写回 HBM
5. 最后算 $O = PV$

所以问题不只是 FLOPs，而是：**中间的 $L \times L$ 矩阵太大，读写 HBM 的代价非常高。**

**FlashAttention 的核心思想：分块 + 在线 softmax。**

把 $Q$、$K$、$V$ 沿序列维切成块。设 $Q$ 的一个块为 $Q_i$，$K$、$V$ 的一个块为 $K_j, V_j$，则局部 score 是：

$$
S_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}}
$$

这里 $S_{ij}$ 只是一个小块，而不是完整的 $L \times L$ 矩阵。FlashAttention 会：

- 把 $Q_i$ 放到片上 SRAM
- 依次加载 $K_j, V_j$
- 只计算当前块的局部 score $S_{ij}$
- 立刻把它并入当前输出块的 softmax 累积结果
- 不把完整 $S$ 或完整 $P$ 落到 HBM

**为什么能不存完整 softmax 还算对？**

因为 softmax 可以按块做数值稳定的在线递推。对某一行，设之前已经处理过若干块后，维护：

- 当前最大值 $m$
- 当前归一化因子 $\ell$
- 当前输出累积 $o$

当读入一个新块 $x$ 时，先求该块行最大值：

$$
m_{\text{new}} = \max(m_{\text{old}}, \max(x))
$$

然后更新归一化项：

$$
\ell_{\text{new}} = e^{m_{\text{old}}-m_{\text{new}}}\ell_{\text{old}} + \sum e^{x-m_{\text{new}}}
$$

输出累积也同步更新：

$$
o_{\text{new}} = e^{m_{\text{old}}-m_{\text{new}}} o_{\text{old}} + \sum e^{x-m_{\text{new}}} V_j
$$

最后这一行的真正输出是：

$$
O_i = \frac{o_{\text{final}}}{\ell_{\text{final}}}
$$

这说明 FlashAttention **不是近似 softmax**，而是通过在线维护最大值和归一化因子，精确恢复标准 attention 的结果。

**局部 score / 分块流程怎么理解：**

- 标准 attention：一次性构造完整 $L \times L$ 的 $S$ 和 $P$
- FlashAttention：只算小块 $S_{ij}$，算完立刻合并到输出，不保留完整 $S$ 和 $P$

所以它省掉的是：

- 完整 score matrix 的显存写回
- 完整 softmax matrix 的显存写回
- 大量 HBM <-> SRAM 往返搬运

**为什么快：**

1. 减少 HBM 访问，而现代 GPU 上 HBM 访问往往比算子 FLOPs 更容易成为瓶颈。
2. 分块后数据复用更好，片上 SRAM 利用率更高。
3. 不显式 materialize $L \times L$ attention matrix，显存占用显著下降。

**但它不是什么：**

- 它不是把 attention 变成线性时间。
- 它不是新的注意力定义。
- 它不是近似 attention（FlashAttention 1/2 的主线是 exact attention）。

**总结：**

FlashAttention 的本质不是改公式，而是改计算顺序和内存访问路径。它仍然在算：

$$
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

但通过 **局部 score + 分块计算 + 在线 softmax**，避免了完整 attention matrix 的物化，因此在现代硬件上显著更快、更省显存。

### 手撕题&维度题

如果面试官让你把 **完整 Transformer** 从输入一路讲到输出，最稳的顺序是：

1. `src/tgt token id -> embedding`
2. `embedding + 位置建模`
3. `encoder -> memory`
4. `decoder -> hidden states`
5. `generator -> logits`

**一、输入到 embedding：**

设：

- source token id：

$$
src \in \mathbb{R}^{B \times S}
$$

- target token id：

$$
tgt \in \mathbb{R}^{B \times T}
$$

经过 embedding 后：

$$
E_{src}(src) \in \mathbb{R}^{B \times S \times d_{model}}
$$

$$
E_{tgt}(tgt) \in \mathbb{R}^{B \times T \times d_{model}}
$$

如果用原论文写法，embedding 常乘：

$$
\sqrt{d_{model}}
$$

然后再加位置编码或用 RoPE 这类位置建模方式。

**二、Encoder 里的 self-attention 维度：**

设 encoder 输入：

$$
X \in \mathbb{R}^{B \times S \times d_{model}}
$$

线性投影：

$$
W_Q, W_K, W_V \in \mathbb{R}^{d_{model} \times d_{model}}
$$

则：

$$
Q, K, V \in \mathbb{R}^{B \times S \times d_{model}}
$$

如果有 $h$ 个头，则每头维度：

$$
d_k = d_{model}/h
$$

reshape 后：

$$
Q, K, V \in \mathbb{R}^{B \times h \times S \times d_k}
$$

score 矩阵：

$$
QK^T \in \mathbb{R}^{B \times h \times S \times S}
$$

softmax 后再乘 $V$，每个头输出仍是：

$$
\mathbb{R}^{B \times h \times S \times d_k}
$$

拼接并过输出投影后回到：

$$
\mathbb{R}^{B \times S \times d_{model}}
$$

再经过 FFN、残差、归一化。多层堆叠后，encoder 最终输出：

$$
memory \in \mathbb{R}^{B \times S \times d_{model}}
$$

**三、Decoder 里的两种 attention：**

设 decoder 当前输入表示为：

$$
Y \in \mathbb{R}^{B \times T \times d_{model}}
$$

1. **Masked Self-Attention**

$$
Q, K, V \in \mathbb{R}^{B \times h \times T \times d_k}
$$

$$
QK^T \in \mathbb{R}^{B \times h \times T \times T}
$$

这里用 `tgt_mask`，通常包含 causal mask，防止当前位置看见未来 token。

2. **Cross-Attention**

这一步最容易问：

- query 来自 decoder 当前状态
- key/value 来自 encoder 的 `memory`

所以：

$$
Q \in \mathbb{R}^{B \times h \times T \times d_k}
$$

$$
K, V \in \mathbb{R}^{B \times h \times S \times d_k}
$$

因此 score 维度是：

$$
QK^T \in \mathbb{R}^{B \times h \times T \times S}
$$

这表示每个目标位置对每个源位置打分。这里常配合 `memory_mask` 屏蔽 source padding。

经过 cross-attention 和 FFN 后，decoder 输出 hidden states：

$$
H \in \mathbb{R}^{B \times T \times d_{model}}
$$

**四、输出层 / generator：**

最后用线性层把每个位置从 hidden state 投影到词表：

$$
W_{out} \in \mathbb{R}^{d_{model} \times V_{tgt}}
$$

于是：

$$
logits \in \mathbb{R}^{B \times T \times V_{tgt}}
$$

这里的 $V_{tgt}$ 是目标词表大小。对最后一维做 softmax，就得到每个目标位置对整个词表的概率分布。

**五、mask 怎么答最稳：**

- `src_mask`：屏蔽 source padding，作用在 encoder self-attention
- `tgt_mask`：屏蔽 target padding + future tokens，作用在 decoder self-attention
- `memory_mask`：屏蔽 source padding，作用在 decoder cross-attention

常见维度可以记成：

$$
src\_mask \sim (B, 1, 1, S)
$$

$$
tgt\_mask \sim (B, 1, T, T)
$$

$$
memory\_mask \sim (B, 1, T, S)
$$

**edding：**

> Transformer 的表达能力不是只靠 attention，而是 embedding + 位置建模 + attention + FFN + 残差 + 归一化 + 输出层 一起构成完整系统。

### 14）为什么 decoder 的输入要 shifted right？

**面试短答：**

因为训练时 decoder 要学习“根据前缀预测下一个 token”。所以会把目标序列右移一位后作为输入，例如用 `<bos> i like` 去预测 `i like apples`。这样当前位置只能利用过去信息，和自回归生成目标一致。

**展开回答：**

设目标序列为 `y_1, y_2, ..., y_T`，训练目标是：

$$
P(y_t \mid y_{<t}, x)
$$

因此 decoder 输入通常构造成：

- `tgt_input = [<bos>, y_1, y_2, ..., y_{T-1}]`
- `tgt_output = [y_1, y_2, ..., y_T]`

这就是 teacher forcing 的标准配对方式。

### 15）Cross-Attention 里 Q / K / V 分别来自哪里？

**面试短答：**

在 cross-attention 中，`Q` 来自 decoder 当前状态，`K` 和 `V` 来自 encoder 输出的 `memory`。也就是 decoder 决定“我要找什么”，encoder 提供“我这里有什么内容”。

**展开回答：**

如果 decoder 当前表示为 `x`，encoder 输出为 `memory`，则：

$$
Q = xW_Q, \quad K = memory W_K, \quad V = memory W_V
$$

因此 score 的 shape 是：

$$
QK^T \in \mathbb{R}^{B \times h \times T \times S}
$$

表示每个目标位置对每个源位置打分。

### 16）为什么训练时和推理时 decoder 的行为不一样？

**面试短答：**

训练时通常用 teacher forcing，一次把整段目标序列喂进去；推理时没有真值目标，只能一步一步自回归生成。两者共享同一个因果约束，所以 decoder 训练时必须加 causal mask，防止看到未来 token。

**展开回答：**

训练时，未来 token 实际上已经在张量里，如果不加 mask，当前位置就会偷看答案，造成训练和推理不一致。推理时未来 token 根本不存在，因此模型必须学会在只看前缀的条件下完成预测。

### 17）为什么现在很多大模型是 decoder-only？

**面试短答：**

因为 decoder-only 用统一的 next-token prediction 目标就能做大规模预训练，而且天然适合聊天、写作、代码生成这类生成任务。它不需要单独一套 encoder，也不需要 cross-attention，结构更统一。

**展开回答：**

encoder-decoder 更适合显式 seq2seq 任务，比如翻译、摘要；decoder-only 更适合“给前缀，继续往后写”。现代 LLM 的主流交互方式正好就是这种统一生成范式。

### 18）Embedding 和输出层有什么关系？什么是 weight tying？

**面试短答：**

输入 embedding 是把 token id 映射到向量，输出层则把 hidden state 映射回词表 logits。两者都和词表有关，所以很多模型会共享这两套权重，这就叫 weight tying。

**展开回答：**

如果 embedding 矩阵记作：

$$
E \in \mathbb{R}^{V \times d_{model}}
$$

输出投影通常是：

$$
W_{out} \in \mathbb{R}^{d_{model} \times V}
$$

weight tying 就是令：

$$
W_{out} = E^T
$$

这样可以减少参数，并让输入和输出使用更一致的词表表示空间。

### 19）为什么 attention 更适合长距离依赖，但长文本还是会遇到瓶颈？

**面试短答：**

因为“依赖路径短”和“计算便宜”是两回事。attention 在关系建模上很直接，任意两位置一跳就能交互；但标准 self-attention 需要处理所有两两位置交互，所以时间和显存通常都是二次增长。

**展开回答：**

如果序列长度为 `L`，attention score 的 shape 是：

$$
L \times L
$$

这意味着长文本里最贵的是 attention matrix 本身，而不是 FFN。因此现代改进经常围绕长序列效率展开，比如稀疏 attention、线性 attention、FlashAttention。

### 20）Attention 是不是替代了全部建模能力？

**面试短答：**

不是。Transformer 的表达能力来自 attention + FFN + 残差 + 归一化 + 位置建模 的组合。只讲 attention 而忽略其它模块，说明理解还不完整。

**展开回答：**

attention 负责 token 间的信息交互，FFN 负责逐位置非线性变换，残差和归一化负责可训练性，位置编码负责顺序信息。缺任何一块，标准 Transformer 的能力都会明显下降。

### 21）常见激活函数有哪些？它们各自起什么作用？

**面试短答：**

常见激活函数包括 `ReLU`、`GELU`、`Sigmoid`、`Tanh`、`Leaky ReLU`、`SiLU/Swish`。它们的共同作用是给网络引入非线性；如果没有激活函数，多层线性层叠加后本质上仍然只是一个线性变换。Transformer 的 FFN 里最常见的是 `ReLU` 和 `GELU`，现代大模型里 `GELU`、`SiLU` 更常见。

**展开回答：**

在 FFN 中常写成：

$$
\mathrm{FFN}(x)=W_2\,\sigma(W_1x+b_1)+b_2
$$

这里的 $\sigma$ 就是激活函数。它的作用不是“产生概率”，而是让模型具备非线性表达能力。常见激活函数可以这样记：

1. **ReLU**

$$
\mathrm{ReLU}(x)=\max(0,x)
$$

作用：正数保留，负数置零，实现简单、计算快，是最经典的激活函数。

2. **Leaky ReLU**

$$
\mathrm{LeakyReLU}(x)=
\begin{cases}
x, & x>0 \\
\alpha x, & x\le 0
\end{cases}
$$

作用：给负半轴保留一个很小斜率，减轻 ReLU 负区间全为 0 的问题。

3. **Sigmoid**

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

作用：把输入压到 $(0,1)$，早期网络里很常见，但在深层主干里容易饱和，现代 Transformer 主体里较少单独使用。

4. **Tanh**

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

作用：把输入压到 $(-1,1)$，零中心，比 Sigmoid 更对称，但同样可能出现饱和。

5. **GELU**

常见近似写法：

$$
\mathrm{GELU}(x)\approx 0.5x\left(1+\tanh\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right)
$$

作用：比 ReLU 更平滑，现代 Transformer 很常见，例如 BERT 系列经常使用它。

6. **SiLU / Swish**

$$
\mathrm{SiLU}(x)=x\cdot\sigma(x)
$$

作用：也是平滑型激活，很多较新的大模型和门控 FFN 会用到。

**一句总结：**

- `ReLU`：经典、快、简单
- `GELU`：更平滑，Transformer 常见
- `SiLU`：现代模型常见
- 它们共同作用：给 FFN 引入非线性，增强表达能力